# 15 · AQUASTAT benchmark

Optional external evidence: compare gross FABLE **requirement** to AQUASTAT **observed irrigation withdrawal** indicator 4475 (km³/yr), retaining missing observations and source flags. Indicator 4260 is net requirement; 4250 is not a substitute for 4475.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
DATA = ROOT / 'data'
RUNS = ROOT / 'runs'

GROSS=RUNS/'accepted-evidence-replay'/'gross-master.csv.gz'
BENCH=DATA/'reference'/'aquastat-4475-standardized.csv'
OUT=RUNS/'aquastat'/'benchmark-4475.csv'

In [ ]:
def compare_aquastat(gross, aqua):
    needed=['country_iso3','year','pathway','gross_withdrawal_Mm3']
    if set(needed)-set(gross):raise ValueError('Gross source columns missing')
    if set(['country_iso3','year','withdrawal_km3'])-set(aqua):
        raise ValueError('Provide normalized AQUASTAT 4475 country/year/withdrawal_km3')
    if aqua.duplicated(['country_iso3','year']).any():
        raise ValueError('Resolve duplicate AQUASTAT observations with source flags first')
    model=gross.groupby(['country_iso3','year','pathway'],as_index=False)['gross_withdrawal_Mm3'].sum()
    a=model.merge(aqua,on=['country_iso3','year'],how='left',validate='many_to_one')
    # 1 km3 equals 1000 Mm3. Zero and missing benchmarks are not evaluated.
    good=a.withdrawal_km3.notna()&(a.withdrawal_km3>0)
    a['aquastat_withdrawal_Mm3']=a.withdrawal_km3*1000
    a['relative_difference']=np.where(good,
        a.gross_withdrawal_Mm3/a.aquastat_withdrawal_Mm3-1,np.nan)
    a['evaluation_status']=np.where(good,'MATCHED_REFERENCE_ONLY','NOT_EVALUATED_MISSING_OR_ZERO')
    return a

In [ ]:
for path in (GROSS,BENCH):
    if not path.is_file():raise FileNotFoundError(path)
comparison=compare_aquastat(pd.read_csv(GROSS),pd.read_csv(BENCH))
print(comparison.groupby(['year','pathway','evaluation_status']).size())
OUT.parent.mkdir(parents=True,exist_ok=True)
if OUT.exists():raise FileExistsError(OUT)
comparison.to_csv(OUT,index=False)